# 02. Model Training and Evaluation

This notebook demonstrates how to train an `E3GNN` model and evaluate its performance.

In [1]:
import torch
import pytorch_lightning as pl
from omegaconf import OmegaConf
from dataclasses import asdict

from data.factory import DatasetFactory
from net.common import Config
from net.e3gnn import E3GNN

c:\Users\byoutifel\Documents\documents\Mandala\mandala-venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Creating a Dataset

We use a `DatasetFactory` to create the training and validation datasets.

In [4]:
# Create a DatasetFactory
factory = DatasetFactory(
    cutoff_radius=6.0,
    l_max=2,
    n_radial=64,
)

# Add snapshots to the factory
factory.add_snapshot(
    "../../data/big/silicon/900K/Si_DM",
    "../../data/big/silicon/900K/info.txt",
    purpose="train",
)
factory.add_snapshot(
    "../../data/big/silicon/2700K/Si_DM",
    "../../data/big/silicon/2700K/info.txt",
    purpose="val",
)

# Create the datasets and the mapper
train_ds, val_ds, mapper = factory.create()

print("Training dataset size:", len(train_ds))
print("Validation dataset size:", len(val_ds))

TypeError: DatasetFactory.__init__() got an unexpected keyword argument 'cutoff_radius'

## Model Initialization

We instantiate an `E3GNN` model with a minimal configuration for this demo.

In [5]:
# Model hyperparameters
cfg = Config(
    hidden_base_dim=32,
    l_max=2,
    num_layers_gnn=2,
)

# OmegaConf config
cfg = OmegaConf.create(
    {"model": asdict(cfg), "training": {"lr": 1e-3}, "logging": {"safety_checks": False}}
)

# Create the model
model = E3GNN(mapper, train_ds.edge_types, cfg)

print(model)

UnsupportedValueType: Value 'dtype' is not a supported primitive type
    full_key: model.dtype
    object_type=dict

## Training the Model

We use the PyTorch Lightning `Trainer` to run the training loop.

In [ ]:
# Create DataLoaders
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1, collate_fn=lambda b: b[0])
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=1, collate_fn=lambda b: b[0])

# Create a Trainer
trainer = pl.Trainer(max_epochs=30, accelerator="cpu", devices=1)

# Run training
trainer.fit(model, train_dl, val_dl)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name     | Type        | Params | Mode 
-------------------------------------------------
0 | node_enc | NodeEncoder | 1.1 K  | train
1 | edge_enc | EdgeEncoder | 14.6 K | train
2 | mp_small | ModuleList  | 21.5 K | train
3 | mp_large | ModuleList  | 10.8 K | train
4 | heads    | ModuleDict  | 10.1 K | train
-------------------------------------------------
58.0 K    Trainable params
0         Non-trainable params
58.0 K    Total params
0.232     Total estimated model params size (MB)
240       Module

Sanity Checking DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 216. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/bartek/casus/mandala/mandala-venv/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 29: 100%|██████████| 1/1 [00:01<00:00,  0.68it/s, v_num=2, train_loss_step=7.62e+5, train_loss_blocks_step=7.260, train_loss_E_step=2.22e+10, train_loss_N_step=5.4e+10, train_abs_error_E_step=1.49e+5, train_abs_error_N_step=2.32e+5, val_loss_step=4.12e+5, val_loss_blocks_step=5.330, val_loss_E_step=1.74e+10, val_loss_N_step=2.38e+10, val_abs_error_E_step=1.32e+5, val_abs_error_N_step=1.54e+5, val_loss_epoch=4.12e+5, val_loss_blocks_epoch=5.330, val_loss_E_epoch=1.74e+10, val_loss_N_epoch=2.38e+10, val_abs_error_E_epoch=1.32e+5, val_abs_error_N_epoch=1.54e+5, train_loss_epoch=7.62e+5, train_loss_blocks_epoch=7.260, train_loss_E_epoch=2.22e+10, train_loss_N_epoch=5.4e+10, train_abs_error_E_epoch=1.49e+5, train_abs_error_N_epoch=2.32e+5]  

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 1/1 [00:01<00:00,  0.68it/s, v_num=2, train_loss_step=7.62e+5, train_loss_blocks_step=7.260, train_loss_E_step=2.22e+10, train_loss_N_step=5.4e+10, train_abs_error_E_step=1.49e+5, train_abs_error_N_step=2.32e+5, val_loss_step=4.12e+5, val_loss_blocks_step=5.330, val_loss_E_step=1.74e+10, val_loss_N_step=2.38e+10, val_abs_error_E_step=1.32e+5, val_abs_error_N_step=1.54e+5, val_loss_epoch=4.12e+5, val_loss_blocks_epoch=5.330, val_loss_E_epoch=1.74e+10, val_loss_N_epoch=2.38e+10, val_abs_error_E_epoch=1.32e+5, val_abs_error_N_epoch=1.54e+5, train_loss_epoch=7.62e+5, train_loss_blocks_epoch=7.260, train_loss_E_epoch=2.22e+10, train_loss_N_epoch=5.4e+10, train_abs_error_E_epoch=1.49e+5, train_abs_error_N_epoch=2.32e+5]



## Making Predictions

We can use the trained model to make predictions on a new snapshot.

In [ ]:
# Get a sample from the validation set
x, y_true = val_ds[0]

# Make a prediction
y_pred = model(x)

# Compare the predicted energy to the true energy
true_energy = y_true["energy"].item()
pred_snap = model.predictions_to_snapshot(y_pred, x["positions"], x["box"])
pred_energy = pred_snap.get_energy().item()

print(f"True energy: {true_energy:.4f} eV")
print(f"Predicted energy: {pred_energy:.4f} eV")

True energy: -257.5168 eV
Predicted energy: 131467.7656 eV
